# gVXR aluminium sphere: Beer–Lambert physics validation

This notebook checks one projection against a known analytical result. A monochromatic beam passes through a centered aluminium sphere. At the central detector pixel, the ray crosses the sphere’s diameter, so the expected attenuation is **μ × diameter**. Here μ is calculated independently from xraylib’s mass attenuation coefficient and the aluminium density.

The notebook reports the OpenGL renderer as well: GPU versus software rendering affects performance, while the physics comparison is the same.

## 1. Provide the simulation core
Upload the project’s current `ct_core.py` if it is not already present in the Colab working directory.

In [ ]:
from pathlib import Path
PROJECT_DIR = Path('/content')
if not (PROJECT_DIR / 'ct_core.py').exists():
    from google.colab import files
    uploaded = files.upload()  # upload ct_core.py
    if 'ct_core.py' not in uploaded:
        raise FileNotFoundError('Please upload ct_core.py')


## 2. Install dependencies
Run once in this Colab runtime. `ct_core.py` configures the best available EGL/OpenGL renderer before creating the gVXR context.

In [ ]:
import subprocess, sys
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq',
                'build-essential', 'libxcb-res0', 'libxcb-xinerama0', 'libxcb-xkb1', 'libxcb-icccm4', 'libxcb-image0',
                'libxcb-keysyms1', 'libxcb-render-util0', 'libxcb-shape0', 'libxcb-randr0', 'libxkbcommon-x11-0',
                'libxrandr2', 'libxinerama1', 'libxcursor1', 'libxi6', 'libxxf86vm1',
                'libgl1', 'libegl1', 'libopengl0', 'libglu1-mesa', 'libgl1-mesa-dri', 'libegl-mesa0', 'libglx-mesa0',
                'libxcb-composite0', 'libxcb-cursor0', 'libxcb-dri2-0', 'libxcb-ewmh2',
                'libxcb-dri3-0', 'libxcb-glx0', 'libxcb-present0', 'libxcb-sync1', 'libxcb-xfixes0', 'libxcb-xinput0',
                'libxcb-util1', 'libxcb-xv0', 'libxcb-damage0', 'libxcb-dpms0', 'libxcb-record0', 'libxcb-screensaver0',
                'libxcb-shm0', 'libxcb-xtest0', 'libxcb-xf86dri0', 'libxcb-xvmc0'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gvxr', 'numpy', 'trimesh', 'xraylib', 'matplotlib'], check=True)


## 3. Set up a centered aluminium sphere
The odd number of detector columns gives a pixel centered on the beam axis. A 0.1 mm detector pitch provides fine sampling around the sphere’s silhouette.

In [ ]:
import json, os, platform, sys, tempfile
import numpy as np
import trimesh
import xraylib
import matplotlib.pyplot as plt
sys.path.insert(0, str(PROJECT_DIR.resolve()))
import ct_core as core

# Material parameters
RADIUS_MM = 5.0
DIAMETER_MM = 2 * RADIUS_MM
DENSITY_G_CM3 = 2.70
ENERGY_KEV = 80.0  # monochromatic

# Detector and cone-beam setup
NX, NY = 401, 3  # odd NX puts a pixel at the central ray
PIXEL_MM = 0.1
SOD_CM, ODD_CM = 40.0, 10.0
MAGNIFICATION = (SOD_CM + ODD_CM) / SOD_CM

mesh_path = Path(tempfile.gettempdir()) / 'gvxr_aluminium_validation_sphere.stl'
trimesh.creation.icosphere(subdivisions=4, radius=RADIUS_MM).export(mesh_path)
obj = core.ObjectCfg(name='aluminium_sphere', stl_path=str(mesh_path),
                     kind='Element', material='Al', density=DENSITY_G_CM3)
cfg = core.ScanCfg(objects=[obj], stl_unit='mm', beam='Cone',
                   source_x_cm=SOD_CM, detector_x_cm=ODD_CM,
                   energy_kev=ENERGY_KEV, photons=10000,
                   nx=NX, ny=NY, pixel_mm=PIXEL_MM,
                   n_angles=1, total_deg=360.0, noise=False, spectrum=None)

flat = core.on_gl_thread(core.build_scene, cfg)
raw = core.on_gl_thread(lambda: np.asarray(core._gx().computeXRayImage(), dtype=np.float32))
print('Projection shape:', raw.shape)
print('OpenGL renderer:', core.renderer_info())
print('Renderer selection:', core.renderer_selection_info())


## 4. Compare measured attenuation with μ × path length
For a monoenergetic beam, the Beer–Lambert model predicts `A = −ln(I/I₀) = μL`. At the central pixel, `L` is the sphere diameter. The full line profile is compared with the sphere’s analytical chord length at each detector offset.

In [ ]:
# xraylib returns mass attenuation in cm^2/g; multiply by density to get linear attenuation in cm^-1.
mass_mu_cm2_g = xraylib.CS_Total(xraylib.SymbolToAtomicNumber('Al'), ENERGY_KEV)
mu_cm_inv = mass_mu_cm2_g * DENSITY_G_CM3
diameter_cm = DIAMETER_MM / 10.0
expected_center = mu_cm_inv * diameter_cm

# Flat field is incident intensity I0; raw image is transmitted intensity I.
floor = np.finfo(np.float32).tiny
atten = -np.log(np.maximum(raw, floor) / np.maximum(flat, floor))
row = NY // 2
center_col = NX // 2
measured_center = float(atten[row, center_col])
absolute_error = measured_center - expected_center
relative_error_pct = 100.0 * absolute_error / expected_center

# Convert detector coordinates to object-plane coordinates using magnification.
detector_x_mm = (np.arange(NX) - center_col) * PIXEL_MM
object_x_mm = detector_x_mm / MAGNIFICATION
chord_mm = np.where(np.abs(object_x_mm) <= RADIUS_MM,
                    2.0 * np.sqrt(np.maximum(RADIUS_MM**2 - object_x_mm**2, 0.0)), 0.0)
ideal_profile = mu_cm_inv * (chord_mm / 10.0)

print(f'Material: aluminium, density = {DENSITY_G_CM3:.3f} g/cm^3')
print(f'Energy: {ENERGY_KEV:.1f} keV (monochromatic)')
print(f'Mass attenuation coefficient: {mass_mu_cm2_g:.6f} cm^2/g')
print(f'Linear attenuation coefficient μ: {mu_cm_inv:.6f} cm^-1')
print(f'Central path length: {DIAMETER_MM:.3f} mm = {diameter_cm:.4f} cm')
print(f'Analytical μ × diameter: {expected_center:.6f}')
print(f'Measured −ln(I/I0), central pixel: {measured_center:.6f}')
print(f'Central-pixel relative error: {relative_error_pct:+.3f}%')

plt.figure(figsize=(10, 5))
plt.plot(object_x_mm, atten[row], 'o', ms=3, label='gVXR measured −ln(I/I₀)')
plt.plot(object_x_mm, ideal_profile, '-', lw=2, label='Analytical μ × sphere chord')
plt.axvline(0, color='gray', lw=1, alpha=0.5)
plt.xlabel('Object-space offset from sphere center (mm)')
plt.ylabel('Attenuation, −ln(I/I₀)')
plt.title(f'Aluminium sphere at {ENERGY_KEV:g} keV: central-ray Beer–Lambert check')
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()


## 5. Save the validation result
The JSON records the material, geometry, independent xraylib reference, measured central attenuation, renderer, and relative error. Compare the measured value with the analytical value; small deviations can arise from mesh tessellation, raster sampling, or library/material model differences.

In [ ]:
result = {
    'validation': 'monoenergetic Beer-Lambert central ray through aluminium sphere',
    'material': 'Al', 'density_g_cm3': DENSITY_G_CM3, 'energy_keV': ENERGY_KEV,
    'sphere_diameter_mm': DIAMETER_MM, 'central_path_length_cm': diameter_cm,
    'mass_attenuation_cm2_g_xraylib': float(mass_mu_cm2_g),
    'linear_attenuation_cm_inv': float(mu_cm_inv),
    'expected_mu_times_path': float(expected_center),
    'measured_minus_ln_I_over_I0': measured_center,
    'relative_error_percent': float(relative_error_pct),
    'geometry': {'beam': cfg.beam, 'SOD_cm': SOD_CM, 'SDD_cm': SOD_CM + ODD_CM,
                 'magnification': MAGNIFICATION, 'detector_pixel_mm': PIXEL_MM,
                 'projection_shape_y_x': list(raw.shape)},
    'renderer': core.renderer_info(),
    'renderer_selection': core.renderer_selection_info(),
    'runtime': {'python': sys.version.split()[0], 'platform': platform.platform()}
}
out = Path('gvxr_aluminium_beer_lambert_validation.json')
out.write_text(json.dumps(result, indent=2))
print(out.read_text())
try:
    from google.colab import files
    files.download(str(out))
except ImportError:
    print('Saved locally:', out.resolve())
